# Kaggriculture V40 — Frontier Distillation / Fertilizer Flywheel

**Goal:** leave the V32 local optimum and test a stronger closed-loop economic backbone, then promote a novel derivative only if it survives exact V32 + frontier-parent + public-zoo gates.

## Kaggle settings
- **Accelerator:** None / CPU
- **Internet:** ON
- **Run:** Save Version → Save & Run All

## Required input
Attach the saved output containing:
- `SUBMIT_V32_RUNTIME_VERIFIED.tar.gz`

## Strongly recommended public-agent inputs
- `kaggriculture-frontier-the-soil-remembers-rain`
- `adaptive-farming-strategy-for-kaggriculture`
- `kaggriculture-rank-your-agent`
- `3094-score-kaggriculture`
- `v16-rc5-high-score-8c-4s-premium-market-lead`
- `kaggriculture-frontier-the-moon-counts-melons`
- current public `WEED-Slip` / frontier output if available

The notebook intentionally refuses to create an upload tar if the novel child fails the held-out promotion/live-probe gate.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil
import pandas as pd

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/v40_frontier_distillation')
REPO = Path('/kaggle/working/kaggriculture_v40_source')
PIN = '9c2b1a67efb96b22ffac7996a9e350127f0eeedb'

print('python:', sys.version)
print('cpu:', os.cpu_count())
print('input root:', INPUT, INPUT.exists())
print('pinned V40 commit:', PIN)

v32 = list(INPUT.rglob('SUBMIT_V32_RUNTIME_VERIFIED.tar.gz'))
if not v32:
    v32 = list(INPUT.rglob('SUBMIT_V32_PREMIUM_FRONT_SINGLEFILE.tar.gz'))
assert v32, 'STOP: attach the exact V32 runtime-verified archive.'
print('V32:', v32[0])


In [ ]:
PATTERNS = {
    'soil':'kaggriculture-frontier-the-soil-remembers-rain',
    'adaptive':'adaptive-farming-strategy-for-kaggriculture',
    'ranker':'kaggriculture-rank-your-agent',
    'score3094':'3094-score-kaggriculture',
    'v16':'v16-rc5-high-score-8c-4s-premium-market-lead',
    'melon':'kaggriculture-frontier-the-moon-counts-melons',
    'weed_slip':'weed-slip',
}
rows=[]
for k,p in PATTERNS.items():
    hits=[x for x in INPUT.rglob('*') if p in str(x).lower() and x.name in {'main.py','submission.tar.gz'}]
    rows.append({'family':k,'found':bool(hits),'example':str(hits[0]) if hits else ''})
display(pd.DataFrame(rows))
print('The builder always includes exact V32 and its pinned public frontier parent even if the attached zoo is sparse.')


## Architecture being tested

V40 starts from a pinned MIT-licensed public closed-loop frontier controller rather than V32's mostly fixed route. It then searches six controlled children:

1. **Fertilizer Flywheel** — buy cheap fertilizer only when strawberry shadow value, feed runway, cash, shed headroom and horizon all agree.
2. **Scale 28** — re-open the strawberry/labor ceiling after the new input channel exists.
3. **Scale 32** — additionally test a third quadrant.
4. **Scale 36** — aggressive scale stress test.
5. **Milk Hedge** — a diversified arm for combined-cow milk crashes.
6. **Market Only** — attribution control with no fertilizer buying.

Existing SELL quantities are never changed by the market-risk layer. Only their occupied SELL slots are reordered.

In [ ]:
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git','clone','https://github.com/sidhulyalkar/kaggriculture.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',PIN],check=True)
print('repo ready:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# Ensure the official runtime package exists. Most competition images already include it.
try:
    import kaggle_environments
    print('kaggle_environments:', kaggle_environments.__version__ if hasattr(kaggle_environments,'__version__') else 'available')
except Exception:
    subprocess.run([sys.executable,'-m','pip','install','-q','kaggle-environments'],check=True)


In [ ]:
if WORK.exists():
    shutil.rmtree(WORK)
cmd = [
    sys.executable,
    str(REPO/'scripts'/'v40_frontier_distillation.py'),
    '--input-root', str(INPUT),
    '--work', str(WORK),
    '--repo', str(REPO),
    '--workers', str(min(4, os.cpu_count() or 2)),
]
print('RUN:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
decision = json.loads((WORK/'V40_DECISION.json').read_text())
display(pd.DataFrame([{
    'decision':decision.get('decision'),
    'selected_candidate':decision.get('selected_candidate'),
    'submission_ready':decision.get('submission_ready'),
    'archive_sha256':decision.get('archive_sha256'),
    'reason':decision.get('reason'),
}]))

for fn in ['V40_FINAL_TABLE.csv','heldout_vs_v32.csv','heldout_vs_parent.csv']:
    p=WORK/fn
    if p.exists():
        print('\n',fn)
        display(pd.read_csv(p))


In [ ]:
submission = WORK/'SUBMIT_V40_FRONTIER_DISTILLED.tar.gz'
if decision.get('submission_ready') and submission.exists():
    final = Path('/kaggle/working/SUBMIT_V40_FRONTIER_DISTILLED.tar.gz')
    shutil.copy2(submission, final)
    shutil.copy2(WORK/'V40_DECISION.json', Path('/kaggle/working/V40_DECISION.json'))
    print('V40_FINAL_CONTRACT: PASS')
    print('UPLOAD:', final)
    print('SHA256:', decision['archive_sha256'])
    print('SELECTED:', decision['selected_candidate'])
else:
    print('V40_FINAL_CONTRACT: HOLD')
    print('No V40 tar was promoted. Preserve the submission slot and send back V40_DECISION.json + V40_FINAL_TABLE.csv.')
